# LightGBM LambdaRank Model

Ranking model using LightGBM's `lambdarank` objective, which optimises NDCG directly.

In [ ]:
import sys
from pathlib import Path

cwd = Path(".").resolve()
REPO_ROOT = cwd if (cwd / ".git").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.utils import load_train_cached_sample, train_val_split, compute_ndcg_fast, make_submission
from src.features import build_features

In [ ]:
SAMPLE_FRAC = 0.15
REFRESH_CACHE = False

df = load_train_cached_sample(sample_frac=SAMPLE_FRAC, refresh=REFRESH_CACHE)
n_cols_before = df.shape[1]
df = build_features(df)
print(f"Loaded {len(df):,} rows, {df['srch_id'].nunique():,} searches")
print(f"Columns: {n_cols_before} raw + {df.shape[1] - n_cols_before} engineered = {df.shape[1]} total")

nan_pct = df.isnull().mean()
print(f"\nColumns with NaNs:")
print((nan_pct[nan_pct > 0] * 100).round(1).to_string())

In [ ]:
train_df, val_df = train_val_split(df)

def make_relevance(frame):
    """booking=5, click-only=1, nothing=0"""
    return (frame["booking_bool"] * 5 +
            (frame["click_bool"] - frame["booking_bool"]).clip(lower=0)).values

y_train = make_relevance(train_df)
y_val = make_relevance(val_df)

DROP_COLS = [
    "srch_id", "date_time", "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "random_bool",
]
COMP_RAW = [f"comp{i}_{s}" for i in range(1, 9) for s in ("rate", "inv", "rate_percent_diff")]
drop = set(DROP_COLS + COMP_RAW)
feature_cols = [c for c in train_df.columns if c not in drop]

print(f"Train: {train_df.shape[0]:,}  Val: {val_df.shape[0]:,}")
print(f"Features: {len(feature_cols)}")

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

group_train = train_df.groupby("srch_id").size().values
group_val = val_df.groupby("srch_id").size().values

In [ ]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "num_leaves": 127,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "min_child_samples": 50,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
}

train_set = lgb.Dataset(X_train, label=y_train, group=group_train)
val_set = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set)

model = lgb.train(
    params,
    train_set,
    valid_sets=[val_set],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

In [ ]:
val_df_eval = val_df.copy()
val_df_eval["pred_score"] = model.predict(X_val)

baseline_ndcg = compute_ndcg_fast(val_df_eval, "prop_location_score1")
model_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"Baseline NDCG@5 (location_score1): {baseline_ndcg:.4f}")
print(f"LambdaRank NDCG@5:                 {model_ndcg:.4f}")
print(f"Improvement:                        +{model_ndcg - baseline_ndcg:.4f}")

In [ ]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("Top 20 features by gain:")
print(importance.head(20).to_string(index=False))

## Hyperparameter Search

Grid search over key parameters. Evaluate each on validation NDCG@5.

In [ ]:
from itertools import product

search_space = {
    "num_leaves": [63, 127, 255],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_child_samples": [20, 50, 100],
}

results = []
base_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
}

keys = list(search_space.keys())
for combo in product(*search_space.values()):
    trial_params = {**base_params, **dict(zip(keys, combo))}

    train_set_hp = lgb.Dataset(X_train, label=y_train, group=group_train)
    val_set_hp = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set_hp)

    m = lgb.train(
        trial_params,
        train_set_hp,
        valid_sets=[val_set_hp],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )

    preds = m.predict(X_val)
    val_df_eval["pred_score"] = preds
    score = compute_ndcg_fast(val_df_eval, "pred_score")

    row = dict(zip(keys, combo))
    row["ndcg5"] = score
    row["n_trees"] = m.best_iteration
    results.append(row)
    print(f"  leaves={row['num_leaves']:>3}  lr={row['learning_rate']:.2f}  "
          f"min_child={row['min_child_samples']:>3}  -> NDCG@5={score:.4f}  trees={m.best_iteration}")

results_df = pd.DataFrame(results).sort_values("ndcg5", ascending=False)
print("\nTop 5 configurations:")
print(results_df.head().to_string(index=False))

## Retrain with Best Params

Use the best configuration from the grid search, then save the model.

In [ ]:
best = results_df.iloc[0]
print(f"Best config: leaves={int(best.num_leaves)}, lr={best.learning_rate}, "
      f"min_child={int(best.min_child_samples)}, NDCG@5={best.ndcg5:.4f}")

best_params = {
    **base_params,
    "num_leaves": int(best.num_leaves),
    "learning_rate": best.learning_rate,
    "min_child_samples": int(best.min_child_samples),
}

train_set_best = lgb.Dataset(X_train, label=y_train, group=group_train)
val_set_best = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set_best)

best_model = lgb.train(
    best_params,
    train_set_best,
    valid_sets=[val_set_best],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

val_df_eval["pred_score"] = best_model.predict(X_val)
final_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"\nFinal NDCG@5: {final_ndcg:.4f}")

models_dir = REPO_ROOT / "models"
models_dir.mkdir(exist_ok=True)
best_model.save_model(str(models_dir / "lambdarank_best.txt"))